In [1]:
# IMPORTS
from __future__ import annotations

import re
import pandas as pd
import docker
import json
import resource
import traceback

from typing import cast
from argparse import ArgumentParser
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm

from swebench.harness.constants import (
    APPLY_PATCH_FAIL,
    APPLY_PATCH_PASS,
    INSTANCE_IMAGE_BUILD_DIR,
    KEY_INSTANCE_ID,
    RUN_EVALUATION_LOG_DIR,
    SWEbenchInstance,
)
from swebench.harness.docker_utils import (
    remove_image,
    copy_to_container,
    copy_from_container,
    exec_run_with_timeout,
    cleanup_container,
    list_images,
    should_remove,
    clean_images,
)
from swebench.harness.docker_build import (
    BuildImageError,
    build_container,
    build_env_images,
    close_logger,
    setup_logger,
)

from swebench.harness.run_evaluation import *
from swebench.harness.grading import get_eval_report
from swebench.harness.test_spec import (
    ut_make_test_spec,
    TestSpec,
    get_pytest_commands_from_file,
    extract_added_lines,
)
from swebench.harness.utils import load_swebench_dataset, get_test_directives

/Users/sreekarm/SWE-bench/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-09-11 11:02:36,102 - datasets - INFO - PyTorch version 2.4.0 available.


In [2]:
# DATA
splits = {
    "dev": "data/dev-00000-of-00001.parquet",
    "test": "data/test-00000-of-00001.parquet",
}
df = pd.read_parquet("hf://datasets/princeton-nlp/SWE-bench_Lite/" + splits["dev"])

In [3]:
def ut_run_instance(
    test_spec: TestSpec,
    pred: dict,
    rm_image: bool,
    force_rebuild: bool,
    client: docker.DockerClient,
    run_id: str,
    timeout: int | None = None,
):
    """
    Run a single instance with the given prediction.

    Args:
        test_spec (TestSpec): TestSpec instance
        pred (dict): Prediction w/ model_name_or_path, model_patch, instance_id
        rm_image (bool): Whether to remove the image after running
        force_rebuild (bool): Whether to force rebuild the image
        client (docker.DockerClient): Docker client
        run_id (str): Run ID
        timeout (int): Timeout for running tests
    """
    # Set up logging directory
    instance_id = test_spec.instance_id
    model_name_or_path = pred.get("model_name_or_path", "None").replace("/", "__")
    log_dir = RUN_EVALUATION_LOG_DIR / run_id / model_name_or_path / instance_id
    log_dir.mkdir(parents=True, exist_ok=True)

    # Link the image build dir in the log dir
    build_dir = INSTANCE_IMAGE_BUILD_DIR / test_spec.instance_image_key.replace(
        ":", "__"
    )
    image_build_link = log_dir / "image_build_dir"
    if not image_build_link.exists():
        try:
            # link the image build dir in the log dir
            image_build_link.symlink_to(build_dir.absolute(), target_is_directory=True)
        except:
            # some error, idk why
            pass
    log_file = log_dir / "run_instance.log"

    # Set up report file + logger
    report_path = log_dir / "report.json"
    if report_path.exists():
        return instance_id, json.loads(report_path.read_text())
    logger = setup_logger(instance_id, log_file)

    # Run the instance
    container = None
    try:
        # Build + start instance container (instance image should already be built)
        container = build_container(
            test_spec, client, run_id, logger, rm_image, force_rebuild
        )
        container.start()
        logger.info(f"Container for {instance_id} started: {container.id}")
        ############################################################################################################
        # # Obtain original pytest commands

        # original_test_cmds = []

        # for file in test_spec.test_paths:
        #     test_file_path = Path(log_dir / "test_files" / "base" / file)
        #     copy_from_container(container, file, test_file_path, logger)
        #     original_file_cmds = get_pytest_commands_from_file(test_file_path, test_spec.test_patch)
        #     logger.info(
        #         f"Obtained {file} commands: {original_file_cmds}"
        #     )
        #     original_test_cmds += original_file_cmds

        # copy test patch diff

        patch_file = Path(log_dir / "test_patch.diff")
        patch_file.write_text(test_spec.test_patch)
        logger.info(
            f"Test patch for {instance_id} written to {patch_file}, now applying to container..."
        )
        copy_to_container(container, patch_file, Path("/tmp/test_patch.diff"))

        # Attempt to apply patch to container
        val = container.exec_run(
            "git apply --allow-empty -v /tmp/test_patch.diff",
            workdir="/testbed",
            user="root",
        )
        if val.exit_code != 0:
            logger.info(f"Failed to apply patch to container, trying again...")

            # try "patch --batch --fuzz=5 -p1 -i {patch_path}" to try again
            val = container.exec_run(
                "patch --batch --fuzz=5 -p1 -i /tmp/test_patch.diff",
                workdir="/testbed",
                user="root",
            )
            if val.exit_code != 0:
                logger.info(f"{APPLY_PATCH_FAIL}:\n{val.output.decode('utf-8')}")
                raise EvaluationError(
                    instance_id,
                    f"{APPLY_PATCH_FAIL}:\n{val.output.decode('utf-8')}",
                    logger,
                )
            else:
                logger.info(f"{APPLY_PATCH_PASS}:\n{val.output.decode('utf-8')}")
        else:
            logger.info(f"{APPLY_PATCH_PASS}:\n{val.output.decode('utf-8')}")

        git_diff_output_before = (
            container.exec_run("git diff", workdir="/testbed")
            .output.decode("utf-8")
            .strip()
        )
        logger.info(f"Git diff before running eval script:\n{git_diff_output_before}")

        # obtain created pytest commands
        test_commands = []
        for file in test_spec.test_paths:
            post_test_file_path = Path(log_dir/"test_files"/ file)
            copy_from_container(container, file, post_test_file_path, logger)

            new_test_cmds = get_pytest_commands_from_file(post_test_file_path, test_spec.test_patch)
            logger.info(f"Obtained added pytest commands: {new_test_cmds}")
            test_commands += new_test_cmds

        base_command = "pytest -rA "
        reset_tests_command = (
            f"git checkout {test_spec.base_commit} {' '.join(test_spec.test_files)}"
        )
        pytest_command = base_command + " ".join(test_commands)

        updated_eval_script = test_spec.eval_script_list
        updated_eval_script += [pytest_command]
        updated_eval_script += [reset_tests_command]
        logger.info(f"Finished adding pytest commnads to eval script")

        # Run unit test eval script prefix and write output to logs
        eval_file = Path(log_dir / f"eval.sh")
        eval_file.write_text(
            "\n".join(["#!/bin/bash", "set -uxo pipefail"] + updated_eval_script) + "\n"
        )
        logger.info(
            f"Eval script for {instance_id} written to {eval_file}; copying to container..."
        )
        copy_to_container(container, eval_file, Path("/eval.sh"))

        test_output, timed_out, total_runtime = exec_run_with_timeout(
            container, f"/bin/bash {Path("/eval.sh")}", timeout
        )
        test_output_path = log_dir / "pre_fix_test_output.txt"
        logger.info(f"Test runtime: {total_runtime:_.2f} seconds")
        with open(test_output_path, "w") as f:
            f.write(test_output)
            logger.info(f"Test output for {instance_id} written to {test_output_path}")
            if timed_out:
                f.write(f"\n\nTimeout error: {timeout} seconds exceeded.")
                raise EvaluationError(
                    instance_id,
                    f"Test timed out after {timeout} seconds.",
                    logger,
                )
        ############################################################################################################
        # Copy model prediction as patch file to container
        patch_file = Path(log_dir / "patch.diff")
        patch_file.write_text(pred["model_patch"] or "")
        logger.info(
            f"Intermediate patch for {instance_id} written to {patch_file}, now applying to container..."
        )
        copy_to_container(container, patch_file, Path("/tmp/patch.diff"))

        # Attempt to apply patch to container
        val = container.exec_run(
            "git apply --allow-empty -v /tmp/patch.diff",
            workdir="/testbed",
            user="root",
        )
        if val.exit_code != 0:
            logger.info(f"Failed to apply patch to container, trying again...")

            # try "patch --batch --fuzz=5 -p1 -i {patch_path}" to try again
            val = container.exec_run(
                "patch --batch --fuzz=5 -p1 -i /tmp/patch.diff",
                workdir="/testbed",
                user="root",
            )
            if val.exit_code != 0:
                logger.info(f"{APPLY_PATCH_FAIL}:\n{val.output.decode('utf-8')}")
                raise EvaluationError(
                    instance_id,
                    f"{APPLY_PATCH_FAIL}:\n{val.output.decode('utf-8')}",
                    logger,
                )
            else:
                logger.info(f"{APPLY_PATCH_PASS}:\n{val.output.decode('utf-8')}")
        else:
            logger.info(f"{APPLY_PATCH_PASS}:\n{val.output.decode('utf-8')}")

        # Get git diff before running eval script
        git_diff_output_before = (
            container.exec_run("git diff", workdir="/testbed")
            .output.decode("utf-8")
            .strip()
        )
        logger.info(f"Git diff before running eval script:\n{git_diff_output_before}")
        ############################################################################################################
        # Run eval script post fix

        test_output, timed_out, total_runtime = exec_run_with_timeout(
            container, f"/bin/bash {Path("/eval.sh")}", timeout
        )
        test_output_path = log_dir / "test_output.txt"
        logger.info(f"Test runtime: {total_runtime:_.2f} seconds")
        with open(test_output_path, "w") as f:
            f.write(test_output)
            logger.info(
                f"Test output for {instance_id} written to {test_output_path}"
            )
            if timed_out:
                f.write(f"\n\nTimeout error: {timeout} seconds exceeded.")
                raise EvaluationError(
                    instance_id,
                    f"Test timed out after {timeout} seconds.",
                    logger,
                )
        ############################################################################################################
        # check if git diff changed after running eval script
        git_diff_output_after = (
            container.exec_run("git diff", workdir="/testbed")
            .output.decode("utf-8")
            .strip()
        )

        if git_diff_output_after != git_diff_output_before:
            logger.info(f"Git diff changed after running eval script")

        return instance_id
    except EvaluationError as e:
        error_msg = traceback.format_exc()
        logger.info(error_msg)
        print(e)
    except BuildImageError as e:
        error_msg = traceback.format_exc()
        logger.info(error_msg)
        print(e)
    except Exception as e:
        error_msg = (
            f"Error in evaluating model for {instance_id}: {e}\n"
            f"{traceback.format_exc()}\n"
            f"Check ({logger.log_file}) for more information."
        )
        logger.error(error_msg)
    finally:
        # Remove instance container + image, close logger
        cleanup_container(client, container, logger)
        if rm_image:
            remove_image(client, test_spec.instance_image_key, logger)
        close_logger(logger)
    return

In [4]:
def ut_run_instances(
    predictions: dict,
    instances: list,
    cache_level: str,
    clean: bool,
    force_rebuild: bool,
    max_workers: int,
    run_id: str,
    timeout: int,
):
    """
    Run all instances for the given predictions in parallel.

    Args:
        predictions (dict): Predictions dict generated by the model
        instances (list): List of instances
        cache_level (str): Cache level
        clean (bool): Clean images above cache level
        force_rebuild (bool): Force rebuild images
        max_workers (int): Maximum number of workers
        run_id (str): Run ID
        timeout (int): Timeout for running tests
    """
    client = docker.from_env()
    test_specs = list(map(ut_make_test_spec, instances))

    # print number of existing instance images
    instance_image_ids = {x.instance_image_key for x in test_specs}
    existing_images = {
        tag
        for i in client.images.list(all=True)
        for tag in i.tags
        if tag in instance_image_ids
    }
    if not force_rebuild and len(existing_images):
        print(
            f"Found {len(existing_images)} existing instance images. Will reuse them."
        )

    # run instances in parallel
    print(f"Running {len(instances)} instances...")
    with tqdm(total=len(instances), smoothing=0) as pbar:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Create a future for running each instance
            futures = {
                executor.submit(
                    ut_run_instance,
                    test_spec,
                    predictions[test_spec.instance_id],
                    should_remove(
                        test_spec.instance_image_key,
                        cache_level,
                        clean,
                        existing_images,
                    ),
                    force_rebuild,
                    client,
                    run_id,
                    timeout,
                ): None
                for test_spec in test_specs
            }
            # Wait for each future to complete
            for future in as_completed(futures):
                pbar.update(1)
                try:
                    # Update progress bar, check if instance ran successfully
                    future.result()
                except Exception as e:
                    traceback.print_exc()
                    continue
    print("All instances run.")

In [5]:
def ut_main(
    dataset_name: str,
    split: str,
    instance_ids: list,
    predictions_path: str,
    max_workers: int,
    force_rebuild: bool,
    cache_level: str,
    clean: bool,
    open_file_limit: int,
    run_id: str,
    timeout: int,
):
    """
    Run evaluation harness for the given dataset and predictions.
    """
    # set open file limit
    assert len(run_id) > 0, "Run ID must be provided"
    resource.setrlimit(resource.RLIMIT_NOFILE, (open_file_limit, open_file_limit))
    client = docker.from_env()

    # load predictions as map of instance_id to prediction
    if predictions_path == "gold":
        print("Using gold predictions - ignoring predictions_path")
        predictions = get_gold_predictions(dataset_name, split)
    else:
        if predictions_path.endswith(".json"):
            with open(predictions_path, "r") as f:
                predictions = json.load(f)
        elif predictions_path.endswith(".jsonl"):
            with open(predictions_path, "r") as f:
                predictions = [json.loads(line) for line in f]
        else:
            raise ValueError('Predictions path must be "gold", .json, or .jsonl')
    predictions = {pred[KEY_INSTANCE_ID]: pred for pred in predictions}

    # get dataset from predictions
    dataset = get_dataset_from_preds(
        dataset_name, split, instance_ids, predictions, run_id
    )
    full_dataset = load_swebench_dataset(dataset_name, split, instance_ids)
    existing_images = list_images(client)
    print(f"Running {len(dataset)} unevaluated instances...")
    if not dataset:
        print("No instances to run.")
    else:
        # build environment images + run instances
        build_env_images(client, dataset, force_rebuild, max_workers)
        ut_run_instances(
            predictions,
            dataset,
            cache_level,
            clean,
            force_rebuild,
            max_workers,
            run_id,
            timeout,
        )

    # clean images + make final report
    clean_images(client, existing_images, cache_level, clean)

In [6]:
def ut_load_swebench_dataset(
    name="princeton-nlp/SWE-bench", split="test", instance_ids=None
) -> list[SWEbenchInstance]:
    """
    Load SWE-bench dataset from Hugging Face Datasets or local .json/.jsonl file
    """
    # check that all instance IDs are in the dataset
    if instance_ids:
        instance_ids = set(instance_ids)
    # Load from local .json/.jsonl file
    if name.endswith(".json") or name.endswith(".jsonl"):
        dataset = json.loads(Path(name).read_text())
        dataset_ids = {instance[KEY_INSTANCE_ID] for instance in dataset}
    if instance_ids:
        if instance_ids - dataset_ids:
            raise ValueError(
                (
                    "Some instance IDs not found in dataset!"
                    f"\nMissing IDs:\n{' '.join(instance_ids - dataset_ids)}"
                )
            )
        dataset = [
            instance
            for instance in dataset
            if instance[KEY_INSTANCE_ID] in instance_ids
        ]
    return [cast(SWEbenchInstance, instance) for instance in dataset]

In [7]:
dataset = ut_load_swebench_dataset("/Users/sreekarm/SWE-bench/og_dataset.json", "dev")
run_ids = {i[KEY_INSTANCE_ID] for i in dataset}

ut_main(
    dataset_name="/Users/sreekarm/SWE-bench/og_dataset.json",
    predictions_path="/Users/sreekarm/SWE-bench/og_predictions.json",
    split="dev",
    instance_ids=run_ids,
    max_workers=4,
    force_rebuild=False,
    cache_level="env",
    clean=False,
    open_file_limit=1024,
    run_id="9-11-run1",
    timeout=600,
)

Running 18 unevaluated instances...
Base image sweb.base.arm64:latest already exists, skipping build.
Base images built successfully.
No environment images need to be built.
Running 18 instances...


 33%|███▎      | 6/18 [03:47<07:35, 37.92s/it]

Evaluation error for pvlib__pvlib-python-1154: >>>>> Patch Apply Failed:
can't find file to patch at input line 5
Perhaps you used the wrong -p or --strip option?
The text leading up to this was:
--------------------------
|diff --git a/pvlib/tests/test_irradiance.py b/pvlib/tests/test_irradiance.py
|index dd914eb..7144b9e 100644
|--- a/pvlib/tests/test_irradiance.py
|+++ b/pvlib/tests/test_irradiance.py
--------------------------
No file to patch.  Skipping patch.
1 out of 1 hunk ignored

Check (logs/run_evaluation/9-11-run1/dev/pvlib__pvlib-python-1154/run_instance.log) for more information.
Evaluation error for pvlib__pvlib-python-1707: >>>>> Patch Apply Failed:
can't find file to patch at input line 5
Perhaps you used the wrong -p or --strip option?
The text leading up to this was:
--------------------------
|diff --git a/pvlib/tests/test_iam.py b/pvlib/tests/test_iam.py
|index eba1c66..9479c80 100644
|--- a/pvlib/tests/test_iam.py
|+++ b/pvlib/tests/test_iam.py
-------------------

100%|██████████| 18/18 [10:04<00:00, 33.60s/it]

All instances run.
Cleaning cached images...
Removed 0 images.
